# Instalaciones

In [ ]:
# %pip install jenkspy
# %pip install scikit-posthocs
# %pip install jinja2



## Importaciones

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import spearmanr
import jenkspy
import scikit_posthocs as sp
from scipy.stats import kruskal
import scikit_posthocs as sp



Carga del Dataset

In [ ]:
df = pd.read_csv("../Data/data_cleaning_2026_27_04.csv", decimal=",")

In [ ]:
df.columns

Transformación de datos

In [ ]:
df_clean = df.loc[:, ~df.columns.isin(['mascota', 'peso', 'estatura'])]

In [ ]:
df_clean = df_clean[df_clean['horas_ausentismo'] > 0]
df_clean.info()

## Tener hijos incide su tiempo de ausencia?   
Tests no paramétricos
La variable horas_ausentismo presenta asimetría positiva (muchas ausencias cortas, pocas muy largas), por lo que no sigue distribución normal. Los tests paramétricos clásicos (t-test, ANOVA, Pearson) asumen normalidad y son sensibles a outliers — en este dataset darían resultados poco fiables. Se sustituyeron por sus equivalentes no paramétricos: Mann-Whitney U para comparar dos grupos.

In [ ]:
con_hijos = df_clean[df_clean['hijos'] > 0]['horas_ausentismo']
sin_hijos = df_clean[df_clean['hijos'] == 0]['horas_ausentismo']

stat, p = stats.mannwhitneyu(con_hijos, sin_hijos, alternative='two-sided')
print(f"p-valor: {p:.4f}")
print("Diferencia significativa" if p < 0.05 else "Sin diferencia significativa")

print(f"\nMediana con hijos:  {con_hijos.median():.1f} h")
print(f"Mediana sin hijos:  {sin_hijos.median():.1f} h")

In [ ]:
corr, p = spearmanr(df_clean['hijos'], df_clean['horas_ausentismo'])
print(f"Correlación Spearman: {corr:.3f}")
print(f"p-valor: {p:.4f}")
print("Relación significativa" if p < 0.05 else "Sin relación significativa")

---

## ¿El rango de edad influye en la duración de los episodios de ausencia?   
El rango de edad: el metodo para encontrar los cortes donde la variación dentro del grupo es mínima y entre grupos es máxima. Diseñado para agrupar una variable continua de forma objetiva: agrupación de edad con Jenks Natural Breaks  
Para aplicar Kruskal-Wallis, edad necesita convertirse en grupos discretos. Definir los cortes manualmente (ej. 18-30, 31-45...) introduce sesgo del analista: los resultados del test dependerían de una decisión arbitraria, no de la estructura real de los datos. Se evaluaron tres alternativas: rangos manuales (descartado por sesgo), KMeans (descartado porque está diseñado para espacios multidimensionales y es innecesariamente complejo para una sola variable), y Jenks Natural Breaks. Este último algoritmo encuentra los cortes donde la variación dentro de cada grupo es mínima y la variación entre grupos es máxima — es decir, los grupos resultantes son internamente homogéneos y externamente distintos. Es el método estándar en análisis de una variable continua cuando no se tiene conocimiento previo del dominio para justificar cortes específicos

In [ ]:
# 1. Crear grupos de edad con Jenks Natural Breaks
breaks = jenkspy.jenks_breaks(df_clean['edad'], n_classes=4)
df_clean['rango_edad'] = pd.cut(df_clean['edad'], bins=breaks, include_lowest=True)

print("Distribución de grupos:")
print(df_clean['rango_edad'].value_counts().sort_index())

# 2. Kruskal-Wallis
grupos = [g['horas_ausentismo'].values
          for _, g in df_clean.groupby('rango_edad', observed=True)]

stat, p = stats.kruskal(*grupos)
print(f"\nKruskal-Wallis — p-valor: {p:.4f}")

# 3. Post-hoc Dunn solo si hay diferencia significativa
if p < 0.05:
    print("\nDiferencia significativa — test de Dunn (Bonferroni):")
    print(sp.posthoc_dunn(df_clean, val_col='horas_ausentismo',
                          group_col='rango_edad', p_adjust='bonferroni'))
else:
    print("Sin diferencia significativa entre grupos de edad.")

---

In [ ]:
df_clean['carga_trabajo_diaria'].head()

In [ ]:
df_clean['carga_trabajo_diaria'] = pd.to_numeric(df_clean['carga_trabajo_diaria'], errors='coerce').astype(int)

## ¿A mayor carga de trabajo diaria, más horas de ausencia?

In [ ]:
# ------------------------------------------------------------
# 1. Test no paramétrico: Correlación de Spearman
# ------------------------------------------------------------

rho, pval = spearmanr(
    df_clean['carga_trabajo_diaria'],
    df_clean['horas_ausentismo']
)

print(f"Coeficiente Spearman: {rho:.4f}")
print(f"p-valor: {pval:.4f}")

if pval < 0.05:
    print("→ Relación monotónica significativa entre carga de trabajo y absentismo.")
else:
    print("→ No se detecta relación significativa entre carga de trabajo y absentismo.")

# ------------------------------------------------------------
# 2. Visualización: Scatterplot
# ------------------------------------------------------------
plt.figure(figsize=(8,5))
sns.regplot(
    data=df_clean,
    x='carga_trabajo_diaria',
    y='horas_ausentismo',
    scatter_kws={'alpha':0.4},
    line_kws={'color':'red'}
)
plt.title("Carga de trabajo diaria vs Horas de absentismo")
plt.xlabel("Carga media diaria")
plt.ylabel("Horas de absentismo")
plt.show()

# ------------------------------------------------------------
# 3. Agrupación de carga de trabajo (Jenks Natural Breaks)
# ------------------------------------------------------------

breaks = jenkspy.jenks_breaks(
    df_clean['carga_trabajo_diaria'], 
    n_classes=4
)

df_clean['grupo_carga'] = pd.cut(
    df_clean['carga_trabajo_diaria'],
    bins=breaks,
    include_lowest=True
)

print("Distribución de grupos de carga:")
print(df_clean['grupo_carga'].value_counts().sort_index())

# ------------------------------------------------------------
# 4. Kruskal-Wallis entre grupos de carga
# ------------------------------------------------------------

grupos = [
    g['horas_ausentismo'].values
    for _, g in df_clean.groupby('grupo_carga', observed=True)
]

stat, p = kruskal(*grupos)
print(f"\nKruskal-Wallis p-valor: {p:.4f}")

if p < 0.05:
    print("→ Diferencias significativas entre grupos de carga.")
    print("\nPost-hoc Dunn (Bonferroni):")
    print(
        sp.posthoc_dunn(
            df_clean,
            val_col='horas_ausentismo',
            group_col='grupo_carga',
            p_adjust='bonferroni'
        )
    )
else:
    print("→ No se detectan diferencias significativas entre grupos de carga.")

In [ ]:
# ============================================================
# Gráfico de Barras: Media de horas de absentismo por grupo de carga
# ============================================================

plt.figure(figsize=(10,5))

# Calculamos la media de absentismo por grupo
media_abs = df_clean.groupby('grupo_carga')['horas_ausentismo'].mean()

sns.barplot(
    x=media_abs.index.astype(str),
    y=media_abs.values,
    palette="Reds"
)

plt.title("Horas de absentismo promedio por grupo de carga de trabajo")
plt.xlabel("Grupo de carga (Jenks)")
plt.ylabel("Horas de absentismo promedio")

# Añadimos etiquetas numéricas encima de cada barra
for i, v in enumerate(media_abs.values):
    plt.text(i, v + 0.5, f"{v:.1f}", ha='center', fontweight='bold')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Boxplot Final: Horas de absentismo por grupo de carga
# ============================================================

plt.figure(figsize=(10,5))

sns.boxplot(
    data=df_clean,
    x='grupo_carga',
    y='horas_ausentismo',
    palette="Reds"
)

plt.title("Distribución de horas de absentismo según grupos de carga de trabajo")
plt.xlabel("Grupo de carga (Jenks)")
plt.ylabel("Horas de absentismo")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

¿El rango de edad influye en la duración de los episodios de ausencia?

In [ ]:
df_clean['edad'].describe()

In [ ]:
# ============================================================
# Segmentación de edad en grupos
# ============================================================

df_clean['grupo_edad'] = pd.cut(
    df_clean['edad'],
    bins=[18, 30, 40, 50, 60],
    labels=['18-30', '31-40', '41-50', '51-60'],
    include_lowest=True
)

df_clean['grupo_edad'].value_counts()

In [ ]:
# ============================================================
# Boxplot: Horas de absentismo por grupo de edad
# ============================================================

plt.figure(figsize=(10,5))
sns.boxplot(
    data=df_clean,
    x='grupo_edad',
    y='horas_ausentismo',
    palette='Blues'
)

plt.title("Distribución de horas de absentismo según grupos de edad")
plt.xlabel("Grupo de edad")
plt.ylabel("Horas de absentismo")
plt.show()

In [ ]:
# ============================================================
# Test Kruskal-Wallis: diferencias entre grupos de edad
# ============================================================

from scipy.stats import kruskal

grupos = [
    df_clean[df_clean['grupo_edad'] == g]['horas_ausentismo']
    for g in df_clean['grupo_edad'].unique()
]

stat, p = kruskal(*grupos)

print(f"p-valor Kruskal-Wallis: {p:.4f}")
print("Diferencias significativas" if p < 0.05 else "Sin diferencias significativas")

In [ ]:
# ============================================================
# Post-hoc Dunn con corrección Bonferroni
# ============================================================

import scikit_posthocs as sp

print(sp.posthoc_dunn(
    df_clean,
    val_col='horas_ausentismo',
    group_col='grupo_edad',
    p_adjust='bonferroni'
))

#dunn

In [ ]:
# ============================================================
# Gráfico de barras: media de absentismo por grupo de edad
# ============================================================

media_edad = df_clean.groupby('grupo_edad')['horas_ausentismo'].mean()

plt.figure(figsize=(8,4))
sns.barplot(
    x=media_edad.index,
    y=media_edad.values,
    palette='Blues'
)

plt.title("Horas promedio de absentismo por grupo de edad")
plt.xlabel("Grupo de edad")
plt.ylabel("Horas promedio")

for i, v in enumerate(media_edad.values):
    plt.text(i, v + 0.5, f"{v:.1f}", ha='center', fontweight='bold')

plt.show()

In [ ]:
# ============================================================
# Tabla final: número de empleados, suma y media por grupo de edad
# ============================================================

tabla_edad = df_clean.groupby('grupo_edad').agg(
    empleados_unicos = ('id', 'nunique'),
    suma_horas = ('horas_ausentismo', 'sum'),
    media_horas = ('horas_ausentismo', 'mean')
).reset_index()

# Redondear la media a 2 decimales
tabla_edad['media_horas'] = tabla_edad['media_horas'].round(2)

tabla_edad

# ¿Los empleados con el cumplimiento_objetivo más elevados son los que tienen más horas de ausentismo?

In [ ]:
# ------------------------------------------------------------
# Scatterplot: Cumplimiento del objetivo vs Horas de absentismo
# ------------------------------------------------------------
plt.figure(figsize=(8,5))
sns.regplot(
    data=df_clean,
    x='cumplimiento_objetivo',
    y='horas_ausentismo',
    scatter_kws={'alpha':0.4},
    line_kws={'color':'red'}
)

plt.title("Cumplimiento del objetivo vs Horas de absentismo")
plt.xlabel("Cumplimiento del objetivo (%)")
plt.ylabel("Horas de absentismo")
plt.show()

In [ ]:
# ------------------------------------------------------------
# Correlación Spearman
# ------------------------------------------------------------
from scipy.stats import spearmanr

rho, p = spearmanr(df_clean['cumplimiento_objetivo'], df_clean['horas_ausentismo'])

print(f"Coeficiente Spearman: {rho:.3f}")
print(f"p-valor: {p:.4f}")

if p < 0.05:
    print("Existe correlación significativa")
else:
    print("No existe correlación significativa")

In [ ]:
# ------------------------------------------------------------
# Crear grupos de desempeño (terciles)
# ------------------------------------------------------------
df_clean['grupo_desempeno'] = pd.qcut(
    df_clean['cumplimiento_objetivo'],
    q=3,
    labels=['Bajo', 'Medio', 'Alto']
)

df_clean['grupo_desempeno'].value_counts()

In [ ]:
# ------------------------------------------------------------
# Test Kruskal-Wallis entre grupos de desempeño
# ------------------------------------------------------------
from scipy.stats import kruskal

grupos = [
    df_clean[df_clean['grupo_desempeno'] == g]['horas_ausentismo']
    for g in df_clean['grupo_desempeno'].unique()
]

stat, p = kruskal(*grupos)

print(f"p-valor Kruskal-Wallis: {p:.4f}")
print("Diferencias significativas" if p < 0.05 else "Sin diferencias significativas")

In [ ]:
# ------------------------------------------------------------
# Post-hoc Dunn (Bonferroni)
# ------------------------------------------------------------
import scikit_posthocs as sp

dunn = sp.posthoc_dunn(
    df_clean,
    val_col='horas_ausentismo',
    group_col='grupo_desempeno',
    p_adjust='bonferroni'
)

dunn

In [ ]:
# ------------------------------------------------------------
# Gráfico de barras: horas promedio por grupo de desempeño
# ------------------------------------------------------------
media_desempeno = df_clean.groupby('grupo_desempeno')['horas_ausentismo'].mean()

plt.figure(figsize=(7,4))
sns.barplot(
    x=media_desempeno.index,
    y=media_desempeno.values,
    palette='Blues'
)

plt.title("Horas promedio de absentismo por nivel de desempeño")
plt.xlabel("Nivel de desempeño")
plt.ylabel("Horas promedio")

for i, v in enumerate(media_desempeno.values):
    plt.text(i, v + 0.5, f"{v:.1f}", ha='center', fontweight='bold')

plt.show()

In [ ]:
df_clean.columns

In [ ]:
df_clean.info()

# Test Random Forest

In [ ]:

df_perfil = df_model.groupby('id').agg({
    'horas_ausentismo':      'sum',
    'bebedor_social':        'first',
    'fumador_social':        'first',
    'sanciones_disciplinarias': 'first',
}).reset_index()

df_perfil['nivel_ausentismo'] = pd.cut(
    df_perfil['horas_ausentismo'],
    bins=[0, 8, 24, 72, float('inf')],
    include_lowest=True,
    labels=['Bajo', 'Moderado', 'Alto', 'Critico']
)

df_perfil['riesgo'] = df_perfil['nivel_ausentismo'].isin(
    ['Alto', 'Critico']).astype(int)

print("Distribución riesgo:")
print(df_perfil['riesgo'].value_counts())

# ── 3. FEATURES — stesse variabili del K-Prototypes ──────────
features = [
     'bebedor_social', 'fumador_social', 'sanciones_disciplinarias'
]

X = df_perfil[features]
y = df_perfil['riesgo']

# ── 4. MODELO ────────────────────────────────────────────────
modelo = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',  
    random_state=42,
    max_depth=5
)
modelo.fit(X, y)

# ── 5. IMPORTANCIA DE VARIABLES ──────────────────────────────
importancia = pd.DataFrame({
    'Variable':   features,
    'Importancia': modelo.feature_importances_
}).sort_values(by='Importancia', ascending=False)

# ── 6. VISUALIZACIÓN ─────────────────────────────────────────
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(8, 5))

sns.barplot(data=importancia, x='Importancia', y='Variable',
            palette='Reds_r', ax=ax)
ax.set_title('Variables que predicen el RIESGO DE ALTO AUSENTISMO',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Peso de importancia en el modelo')
ax.set_ylabel('')

plt.tight_layout()
plt.show()

print("\n--- RANKING EXACTO: VARIABLES QUE EXPLICAN EL RIESGO ---")
display(importancia.style.format({'Importancia': '{:.2%}'}))

# ── 7. EXPORTAR RESULTADOS A CSV ─────────────────────────────

# Guardar la tabla de importancia de variables en un archivo CSV
importancia['Importancia'] = importancia['Importancia'].astype(str).str.replace('.', ',')
# importancia.to_csv("RF_Importancia_ABL_2026_04_05.csv", index=False, sep=';', encoding="utf-8-sig")

# print("Archivo CSV generado: RF_Importancia_ABL_2026_04_05.csv")